In [1]:
pip install open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00


In [16]:
import torch
from PIL import Image
import open_clip
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
import torch.nn as nn

In [17]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Initialize the model, preprocessing function and tokenizer

In [14]:
#from clip_zeroshot import build_and_cache_text_features, build_and_cache_image_features, top_k_accuracy, load_cached_features

In [18]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai')
model.eval()  # model in train mode by default
model.to(device)
tokenizer = open_clip.get_tokenizer('ViT-B-16')

open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


### Download and prepare the caltech data

In [20]:
caltech_dataset = datasets.Caltech101(
    root='./data',
    download=True,
    transform=preprocess
)

In [21]:
len(caltech_dataset)

8677

In [22]:
print(caltech_dataset.categories[:50])

['Faces', 'Faces_easy', 'Leopards', 'Motorbikes', 'accordion', 'airplanes', 'anchor', 'ant', 'barrel', 'bass', 'beaver', 'binocular', 'bonsai', 'brain', 'brontosaurus', 'buddha', 'butterfly', 'camera', 'cannon', 'car_side', 'ceiling_fan', 'cellphone', 'chair', 'chandelier', 'cougar_body', 'cougar_face', 'crab', 'crayfish', 'crocodile', 'crocodile_head', 'cup', 'dalmatian', 'dollar_bill', 'dolphin', 'dragonfly', 'electric_guitar', 'elephant', 'emu', 'euphonium', 'ewer', 'ferry', 'flamingo', 'flamingo_head', 'garfield', 'gerenuk', 'gramophone', 'grand_piano', 'hawksbill', 'headphone', 'hedgehog']


In [28]:
# Prepare the class names
caltech_class_names = [cls.replace('_', ' ') for cls in caltech_dataset.categories]
print(caltech_class_names[:10])

['Faces', 'Faces easy', 'Leopards', 'Motorbikes', 'accordion', 'airplanes', 'anchor', 'ant', 'barrel', 'bass']


In [25]:
# Create a dictionary of {class: [index_of_item1, index_of_item2....]}
from collections import defaultdict

total_labels = set()
class_to_indices = defaultdict(list)

for idx in range(len(caltech_dataset)):
    _, label = caltech_dataset[idx]
    class_to_indices[label].append(idx)
    total_labels.add(label)

In [29]:
# Sample 16 images for every class
import random
random.seed(42)

few_shot_indices = []

for label, indices in class_to_indices.items():
   sampled = random.sample(indices, min(16, len(indices)))
   few_shot_indices.extend(sampled)

In [30]:
# Build the training set
from torch.utils.data import Subset

few_shot_training_dataset = Subset(caltech_dataset, few_shot_indices)

In [31]:
# Build the evaluation set
all_indices = set(range(len(caltech_dataset)))
eval_indices = list(all_indices - set(few_shot_indices))
eval_dataset = Subset(caltech_dataset, eval_indices)

In [32]:
print(len(few_shot_training_dataset))
print(len(eval_dataset))

1616
7061


### Build the prompt learner

In [33]:
tokens = tokenizer(["a photo of a dog"]).to(device)
print(tokens.shape)   # confirm: token IDs, e.g. (1, 77)

embedded = model.token_embedding(tokens)   # try the layer you found
print(embedded.shape)  # should be (1, 77, embed_dim) — e.g. (1, 77, 512)

torch.Size([1, 77])
torch.Size([1, 77, 512])


In [34]:
class PromptLearner(nn.Module):
  def __init__(self, model, n_ctx, tokenizer, ctx_dim, class_names):
    super().__init__()
    placeholder = "X " * n_ctx
    prompts = [f"{placeholder}{name}." for name in class_names]
    tokenized_prompts = tokenizer(prompts)
    self.num_classes = len(class_names)
    with torch.no_grad():
      embedding = model.token_embedding(tokenized_prompts)

    prefix = embedding[:, :1, :]
    suffix = embedding[:, 1 + n_ctx:, :]

    self.register_buffer("prefix", prefix)
    self.register_buffer("suffix", suffix)
    self.register_buffer("tokenized_prompts", tokenized_prompts)
    self.ctx = nn.Parameter(torch.randn(n_ctx, ctx_dim) * 0.02)

  def forward(self):
    ctx = self.ctx.unsqueeze(0).expand(self.num_classes, -1, -1)
    prompts = torch.cat([self.prefix, ctx, self.suffix], dim=1)
    return prompts, self.tokenized_prompts

In [35]:
pl = PromptLearner(model, 4, tokenizer, 512, caltech_class_names)
prompts, tok = pl()